# ESRGAN ×3

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).

SRGAN 과 네 군데가 다르다.

| | SRGAN | ESRGAN |
|---|---|---|
| 생성자 | SRResNet | **RRDB** (BatchNorm 없음) |
| 화소 손실 | MSE | **L1** |
| 지각 손실 | VGG16, 활성화 **이후** | **VGG19 conv5_4, 활성화 이전** |
| 적대적 손실 | 절대 판정 | **RaGAN** (상대 판정) |

## 1. 데이터

In [ ]:
import sys, urllib.request
LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'esrgan_models.py', 'esrgan_losses.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # 이미 불러온 옛 모듈이 남아 있으면 비운다

from sr_utils import *

show_data()        # validation 2패치 + test 2구역

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 16장으로 1 epoch 만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

RaGAN 이 핵심이다. "진짜인가 가짜인가" 대신 **"평균적인 가짜보다 얼마나 더 진짜 같은가"**
를 묻는다. 판별자가 한쪽으로 쏠려도 상대 비교라 신호가 살아남는다.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from esrgan_models import RRDBNetX, Discriminator
from esrgan_losses import ESRGANLoss

N_TRAIN, EPOCHS, BATCH = 16, 1, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
loader = DataLoader(TensorDataset(to_t(lo), to_t(hi)), batch_size=BATCH, shuffle=True)

netG = RRDBNetX(nf=64, nb=8, scale=3).to(dev).train()
netD = Discriminator().to(dev).train()
optG = torch.optim.Adam(netG.parameters(), 1e-4)
optD = torch.optim.Adam(netD.parameters(), 1e-4)
crit = ESRGANLoss().to(dev)

for ep in range(1, EPOCHS + 1):
    gl = dl = dx = dgz = 0.0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        sr = netG(x)
        g_loss = crit.generator(netD(y).detach(), netD(sr), sr, y)   # 1) 생성자
        optG.zero_grad(); g_loss.backward(); optG.step()

        d_real, d_fake = netD(y), netD(sr.detach())
        d_loss = crit.discriminator(d_real, d_fake)                  # 2) 판별자
        optD.zero_grad(); d_loss.backward(); optD.step()

        gl += g_loss.item(); dl += d_loss.item()
        dx += torch.sigmoid(d_real).mean().item()
        dgz += torch.sigmoid(d_fake).mean().item()
    k = len(loader)
    print(f'epoch {ep}/{EPOCHS}  Loss_G {gl/k:.4f}  Loss_D {dl/k:.4f}  '
          f'D(x) {dx/k:.3f}  D(G(z)) {dgz/k:.3f}')

## 3. 학습 로그 — 손실 가중치를 바꾼 지점

두 단계다. **1~100 epoch** 은 논문 기본 가중치, **101~110 epoch** 은 가중치를 재조정해
이어 학습했다. 세로 점선이 그 경계다.

| | pixel (L1) | perceptual (VGG) | gan (RaGAN) |
|---|---|---|---|
| 1~100 epoch | ×0.01 | ×1 | ×0.005 |
| 101~110 epoch | **×1** | **×0.1** | **×0.0005** |

판별자는 두 구간 모두 `D(x)` 와 `D(G(z))` 가 벌어진 채 유지된다. SRGAN 은 epoch 37 에서
무너졌다.

In [ ]:
import pandas as pd

MODEL = f'{BASE}/models/03_esrgan_x3'
e = pd.read_csv(fetch(f'{MODEL}/statistics/train_results.csv', 'log.csv'), index_col=0)
cut = e[e.phase == 1].index.max()          # 가중치를 바꾼 지점

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(e.index, e.d_real, color='#2f6f9f', lw=1.5, label='D(x)  real')
ax[0].plot(e.index, e.d_fake, color='#c96a5b', lw=1.5, label='D(G(z))  fake')
ax[0].set_title('Discriminator output — stays separated')
ax[0].set_ylim(-.05, 1.08); ax[0].legend(fontsize=8)

for k, c, lab in [('w_perceptual', '#d9a441', 'perceptual (VGG)'),
                  ('w_gan', '#c96a5b', 'GAN'), ('w_pixel', '#2f6f9f', 'pixel (L1)')]:
    ax[1].plot(e.index, e[k], color=c, lw=1.5, label=lab)
ax[1].set_yscale('log'); ax[1].set_title('Weighted loss terms'); ax[1].legend(fontsize=8)

ax[2].plot(e.index, e.PSNR, color='#4f9d69', lw=1.5)
ax[2].set_title('Validation PSNR during training'); ax[2].set_ylabel('dB')
for a in ax:
    a.axvline(cut, color='#888', ls='--', lw=1.2)     # 가중치를 바꾼 지점
    a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

for ph, name in ((1, '1~100 epoch  (논문 기본)'), (2, '101~110 epoch (재조정)')):
    r = e[e.phase == ph].iloc[-1]
    tot = r.w_pixel + r.w_perceptual + r.w_gan
    print(f'{name}   Loss_G {r.g_loss:.4f}')
    for k in ('pixel', 'perceptual', 'gan'):
        print(f'    {k:11s}{r["w_" + k] / tot * 100:6.2f}%')

## 4. 결과

In [ ]:
from esrgan_models import load_esrgan

net = load_esrgan(fetch(f'{MODEL}/checkpoints/esrgan_g_x3.pth', 'esrgan_x3.pth'))

@torch.no_grad()
def upscale(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device) / 255
    return (net(t).clamp(0, 1)[0].cpu().numpy().transpose(1, 2, 0) * 255).round().astype('uint8')

show_results(upscale, 'ESRGAN', center=[(180, 300), (300, 80)])

## 5. 평가

In [ ]:
rows = compare(upscale, label='ESRGAN')

PSNR 은 아직 bicubic 아래이고 **SSIM 은 위로 올라왔다.**

| | PSNR | SSIM |
|---|---|---|
| Bicubic | 18.15 | 0.4805 |
| ESRGAN (논문 가중치, 100 epoch) | 16.55 | 0.4208 |
| ESRGAN (재조정, +10 epoch) | **17.87** | **0.4929** |

가중치를 바꾸기 전에는 둘 다 bicubic 보다 낮았다. 화소 손실 가중치가 0.01 이라
Loss_G 의 0.06% 밖에 안 됐고, 옵티마이저가 화소 정확도를 볼 이유가 없었다.
pixel 을 1 로 올리자 그 비중이 33% 가 되면서 10 epoch 만에 PSNR +1.32, SSIM +0.072 올랐다.

**지어내던 질감이 줄어든 것이다.** 아래 인천 결과에서 눈으로 확인할 수 있다.

## 6. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로 확인한다.

In [ ]:
show_test(upscale, 'ESRGAN', center=(800, 800), size=140)